# Operator Overloading in Python

Goal: understand why operator overloading exists, arithmetic and comparison operators, reflected operators, and container-style operators (`len()`, indexing, `in`).

## 1. What is Operator Overloading?

**Why we need it:** you already know `3 + 4` and `"ab" + "cd"` both work with `+`, doing something sensible for each type (the polymorphism notebook covered why). But `+` only knows how to handle the types Python ships with. If you write your own `Vector` class and try `v1 + v2`, Python has no idea what "adding" two vectors should mean — it raises a `TypeError`, because nothing ever told it how.

**Use case:** any custom class where a mathematical or comparison operator has an obvious, natural meaning — adding two vectors, comparing two `Money` amounts, checking whether an item is `in` a custom collection. Without overloading, you'd be stuck writing `v1.add(v2)` instead of `v1 + v2`, or `money1.is_less_than(money2)` instead of `money1 < money2` — technically fine, but noticeably less readable than the operators everyone already knows.

**How to implement:** define a specific double-underscore method on your class, and Python calls it automatically whenever the matching operator is used on your objects. `v1 + v2` is really just Python calling `v1.__add__(v2)` for you — the same mechanism behind `len()` calling `__len__` from the polymorphism notebook, just applied to operators instead of a function call. The cell below shows the `TypeError` you get without any of this, before we start fixing it.

In [1]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y


v1 = Vector(1, 2)
v2 = Vector(3, 4)

try:
    v3 = v1 + v2       # Python has no idea how to add two Vector objects
except TypeError as e:
    print(f"TypeError: {e}")

TypeError: unsupported operand type(s) for +: 'Vector' and 'Vector'


## 2. Arithmetic Operators

**Why we need it:** to fix the exact `TypeError` above, so `v1 + v2` produces a new `Vector` the way you'd expect from basic vector math.

**Use case:** `Vector`, `Money`, `Matrix`, `Fraction` — any class that models a mathematical or measurable quantity, where `+`, `-`, and `*` have an unambiguous, expected meaning.

**How to implement:** define the matching double-underscore method. Each takes `self` and the other operand, and should `return` a new object rather than modifying `self` in place (consistent with how `3 + 4` doesn't change `3`).

| Operator | Method |
|---|---|
| `a + b` | `__add__(self, other)` |
| `a - b` | `__sub__(self, other)` |
| `a * b` | `__mul__(self, other)` |
| `a / b` | `__truediv__(self, other)` |

In [2]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)

    def __sub__(self, other):
        return Vector(self.x - other.x, self.y - other.y)

    def __mul__(self, scalar):
        return Vector(self.x * scalar, self.y * scalar)


v1 = Vector(1, 2)
v2 = Vector(3, 4)

print(v1 + v2)      # calls v1.__add__(v2)
print(v1 - v2)      # calls v1.__sub__(v2)
print(v1 * 3)        # calls v1.__mul__(3)

Vector(4, 6)
Vector(-2, -2)
Vector(3, 6)


## 3. Comparison Operators

**Why we need it:** by default, two objects of a custom class are only ever "equal" if they're the literal same object in memory (Python falls back to comparing by identity). Two separate `Money(10)` objects representing the same amount would compare as *not equal* by default, which is almost never what you actually want, and `<`/`>` don't work at all — Python has no idea how to order two custom objects.

**Use case:** `Money` amounts that should be equal if their value matches (regardless of whether they're the same object), sorting a list of custom objects (`sorted(people, key=...)` needs `<` to work, or you can implement it directly on the class), deduplicating objects in a `set`.

**How to implement:** define the matching double-underscore method, each returning `True`/`False`.

| Operator | Method |
|---|---|
| `a == b` | `__eq__(self, other)` |
| `a != b` | `__ne__` (Python derives this from `__eq__` automatically if you skip it) |
| `a < b` | `__lt__(self, other)` |
| `a <= b` | `__le__(self, other)` |
| `a > b` | `__gt__(self, other)` |
| `a >= b` | `__ge__(self, other)` |

You don't have to define all six — Python can derive some from others in specific contexts (e.g. `sorted()` only needs `__lt__`), but defining `__eq__` and `__lt__` covers the two most common needs.

In [3]:
class Money:
    def __init__(self, amount):
        self.amount = amount

    def __repr__(self):
        return f"Money({self.amount})"

    def __eq__(self, other):
        return self.amount == other.amount

    def __lt__(self, other):
        return self.amount < other.amount


a = Money(10)
b = Money(10)
c = Money(25)

print(a == b)          # True now -- compares by value, not identity
print(a is b)           # still False -- they're two different objects in memory
print(a < c, c < a)

wallet = [Money(50), Money(5), Money(20)]
print(sorted(wallet))   # sorted() only needed __lt__ to work

True
False
True False
[Money(5), Money(20), Money(50)]


## 4. Reflected Operators

**Why we need it:** `v1 * 3` works because Python calls `v1.__mul__(3)` — the object on the *left* gets asked first. But what about `3 * v1`? Now the left operand is a plain `int`, and `int.__mul__` has no idea what a `Vector` is, so it can't handle it. Without something extra, `3 * v1` would fail even though `v1 * 3` works perfectly fine.

**Use case:** anywhere your object might reasonably appear on the *right* side of an operator with a built-in type on the left — `3 * v1`, or `sum([Money(10), Money(20)])`, which internally computes `0 + Money(10) + Money(20)`, starting with a plain `int` `0` on the left.

**How to implement:** define the "reflected" version of the method, named with an `r` prefix — `__radd__` for `+`, `__rmul__` for `*`, and so on. When Python evaluates `3 * v1` and finds that `int.__mul__(3, v1)` doesn't know what to do (it returns a special `NotImplemented` value, not an error, to signal "not my job"), Python automatically retries the *other* way around, calling `v1.__rmul__(3)` instead. You get to reuse your existing multiplication logic — `other * self` is usually the same as `self * other` for the operators that matter here.

In [4]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

    def __mul__(self, scalar):
        return Vector(self.x * scalar, self.y * scalar)

    def __rmul__(self, scalar):
        return self.__mul__(scalar)   # reuse the same logic: order doesn't matter here


v = Vector(2, 3)
print(v * 3)      # v.__mul__(3)
print(3 * v)      # int.__mul__ can't handle this, so Python retries as v.__rmul__(3)


class Money:
    def __init__(self, amount):
        self.amount = amount

    def __repr__(self):
        return f"Money({self.amount})"

    def __add__(self, other):
        other_amount = other.amount if isinstance(other, Money) else other
        return Money(self.amount + other_amount)

    def __radd__(self, other):
        return self.__add__(other)    # makes sum([...]) work, since it starts from 0 + Money(...)


wallet = [Money(10), Money(20), Money(30)]
print(sum(wallet))   # sum() starts with 0 + wallet[0] -- needs Money.__radd__ to handle the 0

Vector(6, 9)
Vector(6, 9)
Money(60)


## 5. Container-style Operators

**Why we need it:** overloading isn't limited to math symbols — `len(x)`, `x[i]`, and `item in x` are also just Python calling a specific double-underscore method behind the scenes, the exact mechanism covered for `len()` in the polymorphism notebook. If your class represents something list-like or collection-like, teaching it these methods lets it behave naturally wherever any built-in sequence would be expected.

**Use case:** a custom `Playlist` class where `len(playlist)`, `playlist[0]`, and `"Song X" in playlist` should all just work, instead of forcing callers to use `playlist.songs[0]` or `playlist.get_song_count()`.

**How to implement:**

| Expression | Method |
|---|---|
| `len(x)` | `__len__(self)` |
| `x[i]` | `__getitem__(self, i)` |
| `item in x` | `__contains__(self, item)` |

In [5]:
class Playlist:
    def __init__(self, songs):
        self.songs = songs

    def __len__(self):
        return len(self.songs)

    def __getitem__(self, index):
        return self.songs[index]

    def __contains__(self, song):
        return song in self.songs


p = Playlist(["Song A", "Song B", "Song C"])

print(len(p))            # calls p.__len__()
print(p[0])                # calls p.__getitem__(0)
print("Song B" in p)      # calls p.__contains__("Song B")
print("Song Z" in p)

for song in p:            # __getitem__ alone is enough to make a class iterable in a for-loop
    print("-", song)

3
Song A
True
False
- Song A
- Song B
- Song C


## 6. Putting It Together: a Full `Money` Class

**Why we need it:** to see arithmetic, comparison, reflected, and readable-printing operators all combined on one realistic class, instead of scattered across separate mini-examples.

**Use case:** a `Money` type that behaves as naturally as a plain number for the operations that make sense — adding two amounts, comparing them, summing a list of them, printing them — while still rejecting nonsense like adding `Money` to a `str`.

**How it's implemented here:** `__repr__` for readable output (from the classes-and-objects notebook), `__add__`/`__sub__` for arithmetic, `__eq__`/`__lt__` for comparison, and `__radd__` so `sum(...)` works. `__add__` explicitly returns `NotImplemented` for anything that isn't `Money` or a number, which lets Python produce its normal, clear `TypeError` instead of your code crashing with a confusing internal error.

In [6]:
class Money:
    def __init__(self, amount, currency="USD"):
        self.amount = amount
        self.currency = currency

    def __repr__(self):
        return f"{self.amount:.2f} {self.currency}"

    def _coerce(self, other):
        if isinstance(other, Money):
            if other.currency != self.currency:
                raise ValueError("cannot combine different currencies")
            return other.amount
        if isinstance(other, (int, float)):
            return other
        return None   # not something we know how to combine with

    def __add__(self, other):
        amount = self._coerce(other)
        if amount is None:
            return NotImplemented   # tells Python "I can't handle this" -> clean TypeError
        return Money(self.amount + amount, self.currency)

    def __radd__(self, other):
        return self.__add__(other)

    def __sub__(self, other):
        amount = self._coerce(other)
        if amount is None:
            return NotImplemented
        return Money(self.amount - amount, self.currency)

    def __eq__(self, other):
        return isinstance(other, Money) and self.amount == other.amount and self.currency == other.currency

    def __lt__(self, other):
        amount = self._coerce(other)
        if amount is None:
            return NotImplemented
        return self.amount < amount


prices = [Money(19.99), Money(5.50), Money(42.00)]

print(prices[0] + prices[1])          # Money + Money
print(prices[0] - Money(10))          # Money - Money
print(sum(prices))                     # uses __radd__ under the hood
print(sorted(prices))                  # uses __lt__ under the hood
print(Money(10) == Money(10))
print(Money(10) == Money(10, "EUR"))   # same amount, different currency -> not equal

try:
    Money(10) + "10 dollars"           # __add__ returns NotImplemented -> Python raises TypeError
except TypeError as e:
    print(f"TypeError: {e}")

25.49 USD
9.99 USD
67.49 USD
[5.50 USD, 19.99 USD, 42.00 USD]
True
False
TypeError: unsupported operand type(s) for +: 'Money' and 'str'
